# A/B testing with repeated events per user

When users are randomized but rows are sessions, ordinary IID resampling treats correlated sessions as independent and can make confidence intervals too narrow. This notebook compares that naive analysis with `bootstrap_two_sample` cluster resampling.

For an end-to-end public experiment, continue with [`06_real_world_ab_hillstrom.ipynb`](06_real_world_ab_hillstrom.ipynb).

In [1]:
import numpy as np

from bootstrapx import bootstrap_two_sample

rng = np.random.default_rng(42)
n_control_users, n_treatment_users, sessions_per_user = 200, 220, 5
control_ids = np.repeat(np.arange(n_control_users), sessions_per_user)
treatment_ids = np.repeat(np.arange(n_treatment_users), sessions_per_user)

control = np.repeat(rng.normal(0.0, 2.0, n_control_users), sessions_per_user)
control += rng.normal(0.0, 0.5, len(control_ids))
treatment = np.repeat(rng.normal(0.3, 2.0, n_treatment_users), sessions_per_user)
treatment += rng.normal(0.0, 0.5, len(treatment_ids))

print(f"{len(control):,} control sessions from {n_control_users} users")
print(f"{len(treatment):,} treatment sessions from {n_treatment_users} users")

1,000 control sessions from 200 users
1,100 treatment sessions from 220 users


In [2]:
iid = bootstrap_two_sample(
    control,
    treatment,
    np.mean,
    method="percentile",
    n_resamples=4_999,
    random_state=0,
)
clustered = bootstrap_two_sample(
    control,
    treatment,
    np.mean,
    control_cluster_ids=control_ids,
    treatment_cluster_ids=treatment_ids,
    method="percentile",
    n_resamples=4_999,
    random_state=0,
)

for name, result in [("Naive IID", iid), ("User cluster", clustered)]:
    ci = result.confidence_interval
    print(
        f"{name:12s}: effect={result.estimate:+.3f}, "
        f"95% CI=[{ci.low:+.3f}, {ci.high:+.3f}], SE={result.standard_error:.3f}"
    )

Naive IID   : effect=+0.326, 95% CI=[+0.153, +0.501], SE=0.088
User cluster: effect=+0.326, 95% CI=[-0.050, +0.703], SE=0.192


Both analyses estimate the same observed event-weighted mean difference. The clustered interval is wider because it recognizes that five sessions from one user do not provide five independent pieces of information.

If the business estimand is an equally weighted user-level outcome, aggregate sessions to one value per user first and run an ordinary independent two-sample comparison. Cluster resampling preserves dependence; it does not automatically change event weighting into user weighting.